# HOT Command Center - Demo Data Setup

Creates the `SB_COMMAND_CENTER` database with all schemas and generates synthetic demo data
with learnable patterns (seasonality, segments, promo correlations).

In [ ]:
CREATE DATABASE IF NOT EXISTS SB_COMMAND_CENTER;

CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.RAW;
CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.FEATURE_STORE;
CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.FORECASTING;
CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.MODELS;
CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.REGISTRY;
CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.SCORING;
CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.SEMANTIC;
CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.AGENTS;
CREATE SCHEMA IF NOT EXISTS SB_COMMAND_CENTER.PROCEDURES;

In [ ]:
USE SCHEMA SB_COMMAND_CENTER.RAW;

-- POS Sales table (modeled after POS_RETAIL_DETAILS_F)
CREATE OR REPLACE TABLE POS_SALES (
    RETAILER_NAME TEXT,
    VENDOR_CODE TEXT,
    UPC TEXT,
    RETAILER_SKU TEXT,
    MANUFACTURER_SKU TEXT,
    STORE_NUMBER TEXT,
    RETAILER_DATE DATE,
    ITEM_DESCRIPTION TEXT,
    ITEM_PRODUCT_LINE TEXT,
    ITEM_BRAND TEXT,
    ITEM_SIZE TEXT,
    GROSS_AMT_SOLD_UNITS NUMBER(12,2),
    GROSS_SALES_RETAIL NUMBER(12,2),
    GROSS_SALES_MARGIN_VALUE NUMBER(12,2),
    NET_SALES_UNITS NUMBER(12,2),
    NET_SALES_RETAIL NUMBER(12,2),
    NET_SALES_COST NUMBER(12,2),
    CUSTOMER_RETURN_UNITS NUMBER(12,2),
    CUSTOMER_RETURN_RETAIL NUMBER(12,2),
    INVENTORY_UNITS NUMBER(12,2),
    INVENTORY_RETAIL NUMBER(12,2),
    ON_ORDER_QUANTITY NUMBER(12,2),
    ONHAND_QUANTITY NUMBER(12,2),
    UNIT_PRICE NUMBER(10,2),
    TOTAL_MARKDOWN NUMBER(12,2),
    RECORD_TYPE TEXT
);

-- Customer transactions (modeled after CUSTOMER_OMNI_V)
CREATE OR REPLACE TABLE CUSTOMER_TRANSACTIONS (
    CUSTOMER_ID TEXT,
    TRANSACTION_ID NUMBER,
    FIRST_NAME TEXT,
    LAST_NAME TEXT,
    STATE TEXT,
    CITY TEXT,
    COUNTRY TEXT,
    ZIP_CODE TEXT,
    EMAIL TEXT,
    TRANSACTION_DATE TIMESTAMP_NTZ,
    BRAND TEXT,
    PURCHASE_AMOUNT FLOAT,
    ORDERED_UNITS NUMBER,
    DISCOUNT_OFFER_APPLIED TEXT,
    PROREWARDS_CUSTOMER TEXT,
    DATASOURCE_ID NUMBER
);

-- Web orders (modeled after WEB_ORDERS_V)
CREATE OR REPLACE TABLE WEB_ORDERS (
    ORDER_ID NUMBER,
    NAME TEXT,
    CREATION_DATE TIMESTAMP_NTZ,
    STATUS TEXT,
    BASE_DISCOUNT_AMOUNT NUMBER(10,2),
    BASE_SHIPPING_AMOUNT FLOAT,
    BASE_TAX_AMOUNT NUMBER(10,2),
    BASE_SUBTOTAL NUMBER(10,2),
    BASE_GRAND_TOTAL NUMBER(10,2),
    COUPON_CODE TEXT,
    STORE_ID NUMBER,
    TOTAL_QTY_ORDERED NUMBER,
    CUSTOMER_ID NUMBER,
    CUSTOMER_EMAIL TEXT,
    CUSTOMER_FIRSTNAME TEXT,
    CUSTOMER_LASTNAME TEXT,
    SHIPPING_STATE TEXT,
    SHIPPING_COUNTRY TEXT
);

-- Web order lines (modeled after WEB_ORDER_LINES_V)
CREATE OR REPLACE TABLE WEB_ORDER_LINES (
    ITEM_ID NUMBER,
    ORDER_ID NUMBER,
    SKU TEXT,
    NAME TEXT,
    BASE_PRICE NUMBER(10,2),
    QTY_ORDERED NUMBER,
    BASE_COST NUMBER(10,2),
    BASE_DISCOUNT_AMOUNT NUMBER(10,2),
    PRODUCT_TYPE TEXT,
    BRAND TEXT,
    PRODUCT_LINE TEXT,
    HAS_ENGRAVING BOOLEAN,
    CREATED_AT TIMESTAMP_NTZ
);

## Generate POS Sales Data (~500K rows)

Patterns injected:
- **Seasonality**: Summer spike for Peak Hydro, Holiday spike for ChefLine
- **Retailer baselines**: Walmart high volume, Kohls lower
- **Brand trends**: Each brand has distinct growth trajectory
- **Markdown lift**: Higher markdowns correlate with higher unit velocity

In [ ]:
INSERT INTO SB_COMMAND_CENTER.RAW.POS_SALES
WITH
retailers AS (
    SELECT column1 AS retailer_name, column2 AS base_volume, column3 AS store_count
    FROM VALUES
        ('Walmart', 80, 350),
        ('Target', 65, 300),
        ('Ulta', 40, 250),
        ('Home Depot', 55, 200),
        ('Kohls', 35, 180)
),
brands AS (
    SELECT column1 AS brand, column2 AS brand_mult, column3 AS summer_affinity, column4 AS holiday_affinity, column5 AS product_line
    FROM VALUES
        ('Peak Hydro', 1.0, 2.0, 1.3, 'Hydration'),
        ('ChefLine', 0.9, 0.8, 2.2, 'Kitchen'),
        ('Ridgeline', 0.7, 1.8, 1.1, 'Outdoor'),
        ('VitaCare', 0.6, 0.5, 1.0, 'Health'),
        ('PrimeLine', 0.5, 0.7, 1.5, 'Personal Care')
),
weeks AS (
    SELECT DATEADD('week', seq4(), '2024-01-06'::DATE) AS retailer_date
    FROM TABLE(GENERATOR(ROWCOUNT => 78))  -- 78 weeks (~18 months)
),
stores AS (
    SELECT r.retailer_name, r.base_volume,
           r.retailer_name || '-S' || LPAD(f.value::TEXT, 4, '0') AS store_number
    FROM retailers r,
         LATERAL FLATTEN(ARRAY_GENERATE_RANGE(1, r.store_count + 1)) f
),
base AS (
    SELECT
        s.retailer_name,
        s.store_number,
        s.base_volume,
        b.brand,
        b.brand_mult,
        b.summer_affinity,
        b.holiday_affinity,
        b.product_line,
        w.retailer_date,
        WEEKOFYEAR(w.retailer_date) AS woy,
        MONTH(w.retailer_date) AS mo
    FROM stores s
    CROSS JOIN brands b
    CROSS JOIN weeks w
),
demand AS (
    SELECT
        *,
        -- Seasonality multiplier
        CASE
            WHEN mo BETWEEN 6 AND 8 THEN summer_affinity
            WHEN mo IN (11, 12) THEN holiday_affinity
            WHEN mo BETWEEN 1 AND 2 THEN 0.7  -- post-holiday slump
            ELSE 1.0
        END AS season_mult,
        -- Markdown (higher in Jan/Jul clearance)
        CASE
            WHEN mo = 1 THEN 0.25
            WHEN mo = 7 THEN 0.20
            WHEN mo IN (11, 12) THEN 0.15
            ELSE 0.05
        END AS markdown_pct,
        -- Random noise seeded by combination
        (0.7 + 0.6 * ABS(MOD(HASH(retailer_name || store_number || brand || retailer_date::TEXT), 10000)) / 10000.0) AS noise
    FROM base
)
SELECT
    retailer_name,
    'HOT' || LPAD(ABS(MOD(HASH(retailer_name), 1000))::TEXT, 4, '0') AS vendor_code,
    LPAD(ABS(MOD(HASH(brand || product_line), 999999999999))::TEXT, 12, '0') AS upc,
    'SKU-' || LEFT(brand, 3) || '-' || LPAD(ABS(MOD(HASH(brand || store_number), 9999))::TEXT, 4, '0') AS retailer_sku,
    'MFG-' || LEFT(brand, 3) || '-001' AS manufacturer_sku,
    store_number,
    retailer_date,
    brand || ' ' || product_line || ' Item' AS item_description,
    product_line AS item_product_line,
    brand AS item_brand,
    'Standard' AS item_size,
    -- Gross units: base * brand * season * noise, with markdown lift
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5), 0) AS gross_amt_sold_units,
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5) * 29.99, 2) AS gross_sales_retail,
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5) * 29.99 * 0.45, 2) AS gross_sales_margin_value,
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5) * 0.95, 0) AS net_sales_units,
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5) * 0.95 * 29.99, 2) AS net_sales_retail,
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5) * 0.95 * 29.99 * 0.55, 2) AS net_sales_cost,
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5) * 0.05, 0) AS customer_return_units,
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5) * 0.05 * 29.99, 2) AS customer_return_retail,
    ROUND(base_volume * brand_mult * season_mult * noise * 3.5, 0) AS inventory_units,
    ROUND(base_volume * brand_mult * season_mult * noise * 3.5 * 29.99, 2) AS inventory_retail,
    ROUND(base_volume * brand_mult * season_mult * noise * 1.2, 0) AS on_order_quantity,
    ROUND(base_volume * brand_mult * season_mult * noise * 3.5, 0) AS onhand_quantity,
    29.99 AS unit_price,
    ROUND(base_volume * brand_mult * season_mult * noise * (1 + markdown_pct * 1.5) * 29.99 * markdown_pct, 2) AS total_markdown,
    'SELLOUT' AS record_type
FROM demand;

## Generate Customer Transactions (~200K rows)

Patterns:
- **Segments**: High-value loyalists (frequent, high spend), Deal-seekers (always use coupons), One-time buyers
- **Brand affinity**: Customers tend to repeat-purchase same brand
- **Geographic clustering**: State-level patterns

In [ ]:
INSERT INTO SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS
WITH
-- 50K customers with segment assignment
customers AS (
    SELECT
        'CUST-' || LPAD(seq4()::TEXT, 6, '0') AS customer_id,
        seq4() AS cust_seq,
        CASE
            WHEN seq4() < 5000 THEN 'loyalist'
            WHEN seq4() < 15000 THEN 'deal_seeker'
            WHEN seq4() < 30000 THEN 'occasional'
            ELSE 'one_time'
        END AS segment,
        CASE
            WHEN seq4() < 5000 THEN 8 + MOD(ABS(HASH(seq4()::TEXT || 'txn')), 13)
            WHEN seq4() < 15000 THEN 4 + MOD(ABS(HASH(seq4()::TEXT || 'txn')), 7)
            WHEN seq4() < 30000 THEN 2 + MOD(ABS(HASH(seq4()::TEXT || 'txn')), 4)
            ELSE 1
        END AS num_transactions,
        CASE MOD(ABS(HASH(seq4()::TEXT)), 5)
            WHEN 0 THEN 'Peak Hydro'
            WHEN 1 THEN 'ChefLine'
            WHEN 2 THEN 'Ridgeline'
            WHEN 3 THEN 'VitaCare'
            ELSE 'PrimeLine'
        END AS primary_brand,
        CASE MOD(ABS(HASH(seq4()::TEXT || 'geo')), 10)
            WHEN 0 THEN 'CA' WHEN 1 THEN 'TX' WHEN 2 THEN 'NY'
            WHEN 3 THEN 'FL' WHEN 4 THEN 'IL' WHEN 5 THEN 'PA'
            WHEN 6 THEN 'OH' WHEN 7 THEN 'GA' WHEN 8 THEN 'NC'
            ELSE 'WA'
        END AS state,
        CASE
            WHEN seq4() < 5000 THEN 'Y'
            ELSE CASE WHEN MOD(seq4(), 20) = 0 THEN 'Y' ELSE 'N' END
        END AS prorewards
    FROM TABLE(GENERATOR(ROWCOUNT => 50000))
),
-- Expand to individual transactions using LATERAL FLATTEN
txn_expanded AS (
    SELECT
        c.customer_id, c.segment, c.primary_brand, c.state, c.prorewards, c.cust_seq,
        f.value::INT AS txn_num
    FROM customers c,
         LATERAL FLATTEN(ARRAY_GENERATE_RANGE(1, c.num_transactions + 1)) f
)
SELECT
    customer_id,
    (cust_seq * 100 + txn_num) AS transaction_id,
    CASE MOD(ABS(HASH(customer_id)), 10)
        WHEN 0 THEN 'James' WHEN 1 THEN 'Mary' WHEN 2 THEN 'John'
        WHEN 3 THEN 'Sarah' WHEN 4 THEN 'Michael' WHEN 5 THEN 'Emily'
        WHEN 6 THEN 'David' WHEN 7 THEN 'Jessica' WHEN 8 THEN 'Chris'
        ELSE 'Ashley'
    END AS first_name,
    CASE MOD(ABS(HASH(customer_id || 'ln')), 8)
        WHEN 0 THEN 'Smith' WHEN 1 THEN 'Johnson' WHEN 2 THEN 'Williams'
        WHEN 3 THEN 'Brown' WHEN 4 THEN 'Jones' WHEN 5 THEN 'Garcia'
        WHEN 6 THEN 'Miller' ELSE 'Davis'
    END AS last_name,
    state,
    CASE state
        WHEN 'CA' THEN 'Los Angeles' WHEN 'TX' THEN 'Houston' WHEN 'NY' THEN 'New York'
        WHEN 'FL' THEN 'Miami' WHEN 'IL' THEN 'Chicago' WHEN 'PA' THEN 'Philadelphia'
        WHEN 'OH' THEN 'Columbus' WHEN 'GA' THEN 'Atlanta' WHEN 'NC' THEN 'Charlotte'
        ELSE 'Seattle'
    END AS city,
    'US' AS country,
    LPAD(ABS(MOD(HASH(customer_id || state), 90000) + 10000)::TEXT, 5, '0') AS zip_code,
    LOWER(CASE MOD(ABS(HASH(customer_id)), 10)
        WHEN 0 THEN 'james' WHEN 1 THEN 'mary' WHEN 2 THEN 'john'
        WHEN 3 THEN 'sarah' WHEN 4 THEN 'michael' WHEN 5 THEN 'emily'
        WHEN 6 THEN 'david' WHEN 7 THEN 'jessica' WHEN 8 THEN 'chris'
        ELSE 'ashley'
    END) || '.' || LOWER(CASE MOD(ABS(HASH(customer_id || 'ln')), 8)
        WHEN 0 THEN 'smith' WHEN 1 THEN 'johnson' WHEN 2 THEN 'williams'
        WHEN 3 THEN 'brown' WHEN 4 THEN 'jones' WHEN 5 THEN 'garcia'
        WHEN 6 THEN 'miller' ELSE 'davis'
    END) || ABS(MOD(HASH(customer_id), 999))::TEXT || '@email.com' AS email,
    -- Transaction date: hash-based spread over 18 months
    DATEADD('day',
        MOD(ABS(HASH(customer_id || txn_num::TEXT || 'date')), 540),
        '2024-01-01'::TIMESTAMP_NTZ
    ) AS transaction_date,
    -- Brand: 70% primary brand, 30% random
    CASE WHEN MOD(ABS(HASH(customer_id || txn_num::TEXT)), 10) < 7 THEN primary_brand
        ELSE CASE MOD(ABS(HASH(customer_id || txn_num::TEXT || 'alt')), 5)
            WHEN 0 THEN 'Peak Hydro' WHEN 1 THEN 'ChefLine' WHEN 2 THEN 'Ridgeline'
            WHEN 3 THEN 'VitaCare' ELSE 'PrimeLine'
        END
    END AS brand,
    -- Purchase amount: segment-driven (hash-based pseudo-random)
    CASE segment
        WHEN 'loyalist' THEN ROUND(50 + MOD(ABS(HASH(customer_id || txn_num::TEXT || 'amt')), 150), 2)
        WHEN 'deal_seeker' THEN ROUND(20 + MOD(ABS(HASH(customer_id || txn_num::TEXT || 'amt')), 60), 2)
        WHEN 'occasional' THEN ROUND(30 + MOD(ABS(HASH(customer_id || txn_num::TEXT || 'amt')), 90), 2)
        ELSE ROUND(15 + MOD(ABS(HASH(customer_id || txn_num::TEXT || 'amt')), 45), 2)
    END AS purchase_amount,
    -- Units
    CASE segment
        WHEN 'loyalist' THEN 2 + MOD(ABS(HASH(customer_id || txn_num::TEXT || 'u')), 4)
        WHEN 'deal_seeker' THEN 1 + MOD(ABS(HASH(customer_id || txn_num::TEXT || 'u')), 4)
        ELSE 1 + MOD(ABS(HASH(customer_id || txn_num::TEXT || 'u')), 3)
    END AS ordered_units,
    -- Discount: deal_seekers almost always have one
    CASE
        WHEN segment = 'deal_seeker' AND MOD(ABS(HASH(customer_id || txn_num::TEXT || 'disc')), 10) < 9
            THEN CASE MOD(ABS(HASH(customer_id || txn_num::TEXT)), 4)
                WHEN 0 THEN 'SAVE20' WHEN 1 THEN 'WELCOME15' WHEN 2 THEN 'FLASH30' ELSE 'LOYALTY10'
            END
        WHEN segment = 'loyalist' AND MOD(ABS(HASH(customer_id || txn_num::TEXT || 'disc')), 10) < 3
            THEN 'LOYALTY10'
        WHEN MOD(ABS(HASH(customer_id || txn_num::TEXT || 'disc')), 20) = 0
            THEN 'WELCOME15'
        ELSE NULL
    END AS discount_offer_applied,
    prorewards AS prorewards_customer,
    1 AS datasource_id
FROM txn_expanded;

## Generate Web Orders (~100K) and Order Lines (~300K)

Patterns:
- **Coupon correlation**: Orders with coupons have higher avg basket size
- **Seasonal promos**: Holiday/summer coupon spikes
- **Geographic clustering**: West coast = more Peak Hydro, Midwest = more ChefLine

In [ ]:
INSERT INTO SB_COMMAND_CENTER.RAW.WEB_ORDERS
WITH
order_gen AS (
    SELECT
        seq4() + 1000000 AS order_id,
        seq4() AS oseq,
        -- Spread over 18 months using hash-based pseudo-random
        DATEADD('minute',
            MOD(ABS(HASH(seq4()::TEXT || 'dt')), 780000),
            '2024-01-01'::TIMESTAMP_NTZ
        ) AS creation_date,
        -- Customer ID (reuse from customer table range)
        MOD(ABS(HASH(seq4()::TEXT || 'cust')), 50000) AS cust_num,
        -- Coupon: seasonal pattern based on derived month
        CASE
            WHEN MONTH(DATEADD('minute', MOD(ABS(HASH(seq4()::TEXT || 'dt')), 780000), '2024-01-01'::TIMESTAMP_NTZ)) IN (11, 12)
                AND MOD(seq4(), 3) = 0 THEN 'HOLIDAY25'
            WHEN MONTH(DATEADD('minute', MOD(ABS(HASH(seq4()::TEXT || 'dt')), 780000), '2024-01-01'::TIMESTAMP_NTZ)) IN (6, 7)
                AND MOD(seq4(), 4) = 0 THEN 'SUMMER20'
            WHEN MOD(seq4(), 8) = 0 THEN 'WELCOME15'
            WHEN MOD(seq4(), 12) = 0 THEN 'FLASH30'
            ELSE NULL
        END AS coupon_code,
        -- Items per order: coupon orders have more items (learnable pattern)
        CASE
            WHEN MOD(seq4(), 3) = 0 OR MOD(seq4(), 8) = 0 OR MOD(seq4(), 12) = 0
                THEN 3 + MOD(ABS(HASH(seq4()::TEXT || 'qty')), 5)
            ELSE 1 + MOD(ABS(HASH(seq4()::TEXT || 'qty')), 4)
        END AS qty,
        -- Geographic
        CASE MOD(ABS(HASH(seq4()::TEXT || 'geo')), 10)
            WHEN 0 THEN 'CA' WHEN 1 THEN 'TX' WHEN 2 THEN 'NY'
            WHEN 3 THEN 'FL' WHEN 4 THEN 'IL' WHEN 5 THEN 'WA'
            WHEN 6 THEN 'OR' WHEN 7 THEN 'CO' WHEN 8 THEN 'AZ'
            ELSE 'MA'
        END AS ship_state
    FROM TABLE(GENERATOR(ROWCOUNT => 100000))
)
SELECT
    order_id,
    'HOT-' || LPAD(order_id::TEXT, 9, '0') AS name,
    creation_date,
    CASE
        WHEN MOD(oseq, 50) = 0 THEN 'canceled'
        WHEN MOD(oseq, 20) = 0 THEN 'processing'
        ELSE 'complete'
    END AS status,
    -- Discount: correlated with coupon
    CASE
        WHEN coupon_code = 'HOLIDAY25' THEN ROUND(qty * 8.50, 2)
        WHEN coupon_code = 'SUMMER20' THEN ROUND(qty * 6.80, 2)
        WHEN coupon_code = 'WELCOME15' THEN ROUND(qty * 5.10, 2)
        WHEN coupon_code = 'FLASH30' THEN ROUND(qty * 10.20, 2)
        ELSE 0
    END AS base_discount_amount,
    ROUND(5.0 + MOD(ABS(HASH(oseq::TEXT || 'ship')), 1000) / 100.0, 2) AS base_shipping_amount,
    ROUND(qty * 2.85, 2) AS base_tax_amount,
    ROUND(qty * 34.99, 2) AS base_subtotal,
    ROUND(qty * 34.99 + (5.0 + MOD(ABS(HASH(oseq::TEXT || 'ship')), 1000) / 100.0) + qty * 2.85
        - CASE
            WHEN coupon_code = 'HOLIDAY25' THEN qty * 8.50
            WHEN coupon_code = 'SUMMER20' THEN qty * 6.80
            WHEN coupon_code = 'WELCOME15' THEN qty * 5.10
            WHEN coupon_code = 'FLASH30' THEN qty * 10.20
            ELSE 0
        END, 2) AS base_grand_total,
    coupon_code,
    MOD(ABS(HASH(oseq::TEXT)), 5) + 1 AS store_id,
    qty AS total_qty_ordered,
    cust_num AS customer_id,
    'customer' || cust_num::TEXT || '@email.com' AS customer_email,
    CASE MOD(cust_num, 10)
        WHEN 0 THEN 'James' WHEN 1 THEN 'Mary' WHEN 2 THEN 'John'
        WHEN 3 THEN 'Sarah' WHEN 4 THEN 'Michael' WHEN 5 THEN 'Emily'
        WHEN 6 THEN 'David' WHEN 7 THEN 'Jessica' WHEN 8 THEN 'Chris'
        ELSE 'Ashley'
    END AS customer_firstname,
    CASE MOD(cust_num, 8)
        WHEN 0 THEN 'Smith' WHEN 1 THEN 'Johnson' WHEN 2 THEN 'Williams'
        WHEN 3 THEN 'Brown' WHEN 4 THEN 'Jones' WHEN 5 THEN 'Garcia'
        WHEN 6 THEN 'Miller' ELSE 'Davis'
    END AS customer_lastname,
    ship_state AS shipping_state,
    'US' AS shipping_country
FROM order_gen;

In [ ]:
INSERT INTO SB_COMMAND_CENTER.RAW.WEB_ORDER_LINES
WITH
orders_ref AS (
    SELECT ORDER_ID, TOTAL_QTY_ORDERED, COUPON_CODE, SHIPPING_STATE, CREATION_DATE
    FROM SB_COMMAND_CENTER.RAW.WEB_ORDERS
),
line_expand AS (
    SELECT
        o.ORDER_ID,
        o.COUPON_CODE,
        o.SHIPPING_STATE,
        o.CREATION_DATE,
        f.value::INT AS line_num
    FROM orders_ref o,
         LATERAL FLATTEN(ARRAY_GENERATE_RANGE(1, o.TOTAL_QTY_ORDERED + 1)) f
)
SELECT
    (ORDER_ID * 10 + line_num) AS item_id,
    ORDER_ID,
    -- SKU: geographic brand affinity (west coast = Peak Hydro, midwest = ChefLine)
    CASE
        WHEN SHIPPING_STATE IN ('CA', 'WA', 'OR', 'CO') AND MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT)), 10) < 5
            THEN 'HF-' || LPAD(MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'sku')), 50)::TEXT, 4, '0')
        WHEN SHIPPING_STATE IN ('IL', 'OH', 'PA', 'MA') AND MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT)), 10) < 4
            THEN 'ChefLine-' || LPAD(MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'sku')), 50)::TEXT, 4, '0')
        ELSE CASE MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'brand')), 5)
            WHEN 0 THEN 'HF-' || LPAD(MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'sku')), 50)::TEXT, 4, '0')
            WHEN 1 THEN 'ChefLine-' || LPAD(MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'sku')), 50)::TEXT, 4, '0')
            WHEN 2 THEN 'OSP-' || LPAD(MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'sku')), 30)::TEXT, 4, '0')
            WHEN 3 THEN 'VCK-' || LPAD(MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'sku')), 20)::TEXT, 4, '0')
            ELSE 'BRN-' || LPAD(MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'sku')), 25)::TEXT, 4, '0')
        END
    END AS sku,
    CASE LEFT(sku, 2)
        WHEN 'HF' THEN 'Peak Hydro ' || CASE MOD(line_num, 3) WHEN 0 THEN 'Wide Mouth 32oz' WHEN 1 THEN 'Standard Mouth 21oz' ELSE 'Coffee Mug 12oz' END
        WHEN 'OX' THEN 'ChefLine ' || CASE MOD(line_num, 3) WHEN 0 THEN 'Good Grips Spatula' WHEN 1 THEN 'Salad Spinner' ELSE 'Pop Container' END
        WHEN 'OS' THEN 'Ridgeline ' || CASE MOD(line_num, 3) WHEN 0 THEN 'Daylite Pack 13L' WHEN 1 THEN 'Farpoint 40' ELSE 'Ultralight Stuff Pack' END
        WHEN 'VC' THEN 'VitaCare ' || CASE MOD(line_num, 3) WHEN 0 THEN 'VapoRub' WHEN 1 THEN 'Humidifier' ELSE 'Thermometer' END
        ELSE 'PrimeLine ' || CASE MOD(line_num, 3) WHEN 0 THEN 'Series 9 Shaver' WHEN 1 THEN 'Ear Thermometer' ELSE 'Hand Blender' END
    END AS name,
    -- Price: varies by brand (hash-based pseudo-random)
    CASE LEFT(sku, 2)
        WHEN 'HF' THEN ROUND(22 + MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'pr')), 33), 2)
        WHEN 'OX' THEN ROUND(12 + MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'pr')), 28), 2)
        WHEN 'OS' THEN ROUND(35 + MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'pr')), 85), 2)
        WHEN 'VC' THEN ROUND(8 + MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'pr')), 22), 2)
        ELSE ROUND(25 + MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'pr')), 55), 2)
    END AS base_price,
    1 AS qty_ordered,
    -- Cost (~55% of price)
    ROUND(base_price * 0.55, 2) AS base_cost,
    -- Discount on line: inherit from coupon
    CASE
        WHEN COUPON_CODE = 'HOLIDAY25' THEN ROUND(base_price * 0.25, 2)
        WHEN COUPON_CODE = 'SUMMER20' THEN ROUND(base_price * 0.20, 2)
        WHEN COUPON_CODE = 'WELCOME15' THEN ROUND(base_price * 0.15, 2)
        WHEN COUPON_CODE = 'FLASH30' THEN ROUND(base_price * 0.30, 2)
        ELSE 0
    END AS base_discount_amount,
    CASE MOD(ABS(HASH(ORDER_ID::TEXT || line_num::TEXT || 'pt')), 4)
        WHEN 0 THEN 'simple'
        WHEN 1 THEN 'configurable'
        WHEN 2 THEN 'simple'
        ELSE 'custom_product'
    END AS product_type,
    -- Brand derived from SKU
    CASE LEFT(sku, 2)
        WHEN 'HF' THEN 'Peak Hydro'
        WHEN 'OX' THEN 'ChefLine'
        WHEN 'OS' THEN 'Ridgeline'
        WHEN 'VC' THEN 'VitaCare'
        ELSE 'PrimeLine'
    END AS brand,
    CASE LEFT(sku, 2)
        WHEN 'HF' THEN 'Hydration'
        WHEN 'OX' THEN 'Kitchen'
        WHEN 'OS' THEN 'Outdoor'
        WHEN 'VC' THEN 'Health'
        ELSE 'Personal Care'
    END AS product_line,
    -- Engraving: only Peak Hydro custom_product type
    CASE WHEN LEFT(sku, 2) = 'HF' AND product_type = 'custom_product' THEN TRUE ELSE FALSE END AS has_engraving,
    CREATION_DATE AS created_at
FROM line_expand;

## Verify Row Counts

In [ ]:
SELECT 'POS_SALES' AS table_name, COUNT(*) AS row_count FROM SB_COMMAND_CENTER.RAW.POS_SALES
UNION ALL
SELECT 'CUSTOMER_TRANSACTIONS', COUNT(*) FROM SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS
UNION ALL
SELECT 'WEB_ORDERS', COUNT(*) FROM SB_COMMAND_CENTER.RAW.WEB_ORDERS
UNION ALL
SELECT 'WEB_ORDER_LINES', COUNT(*) FROM SB_COMMAND_CENTER.RAW.WEB_ORDER_LINES
ORDER BY 1;

## Validate Learnable Patterns

In [ ]:
-- Verify seasonality: Peak Hydro should spike in summer, ChefLine in holiday
SELECT
    ITEM_BRAND,
    CASE
        WHEN MONTH(RETAILER_DATE) BETWEEN 6 AND 8 THEN 'Summer'
        WHEN MONTH(RETAILER_DATE) IN (11, 12) THEN 'Holiday'
        ELSE 'Other'
    END AS season,
    ROUND(AVG(GROSS_AMT_SOLD_UNITS), 1) AS avg_units
FROM SB_COMMAND_CENTER.RAW.POS_SALES
GROUP BY 1, 2
ORDER BY 1, 2;

In [ ]:
-- Verify customer segments have different purchase patterns
SELECT
    CASE
        WHEN CUSTOMER_ID < 'CUST-005000' THEN 'loyalist'
        WHEN CUSTOMER_ID < 'CUST-015000' THEN 'deal_seeker'
        WHEN CUSTOMER_ID < 'CUST-030000' THEN 'occasional'
        ELSE 'one_time'
    END AS segment,
    COUNT(*) AS txn_count,
    ROUND(AVG(PURCHASE_AMOUNT), 2) AS avg_purchase,
    ROUND(SUM(CASE WHEN DISCOUNT_OFFER_APPLIED IS NOT NULL THEN 1 ELSE 0 END)::FLOAT / COUNT(*) * 100, 1) AS pct_with_discount
FROM SB_COMMAND_CENTER.RAW.CUSTOMER_TRANSACTIONS
GROUP BY 1
ORDER BY 1;